# Scryfall Tags: Question Answering  
__Objective:__ Fine tune an LLM to answer questions in the following structure:  
- Q: How would you functionally tag the Magic the Gathering card <card_name>?  
- A: [<tag_1>, <tag_2>, ..., <tag_n>]

## Packages and Data

In [1]:
# packages

## modeling
from transformers import pipeline
from huggingface_hub import notebook_login
from huggingface_hub import Repository
from huggingface_hub import get_full_repo_name

## connect project directory
import sys
from pathlib import Path
dir = str(Path(Path.cwd()).parents[0])
if dir not in sys.path:
    sys.path.append(dir)

## load from project directory
from src.data_gathering.scryfall_qa_dataset import ScryfallQADataset
from src.fine_tuning.training import FineTuneLLM

In [2]:
# login to the hugging face
with open('../huggingface_token.txt', 'r') as f:
    token = f.read()

notebook_login()

In [3]:
# params
from src.config import BUILD_DATASET, TASK, MODEL
from src.config import MAX_INPUT_LENGTH, MAX_TARGET_LENGTH, BATCH_SIZE
from src.config import LEARNING_RATE, GRAD_ACCUMULATION_STEPS, NUM_EPOCHS
from src.config import OUTPUT_DIR

In [4]:
# get dataset
sf = ScryfallQADataset()

## build dataset as needed
if BUILD_DATASET:
    sf.build_dataset(
        task = TASK,
        tag_path = '../reports/scryfall_tags.json',
        train_size_pct = 0.8,
        truncate_dataset = 500,
        test_size_n = 10
    )

## load dataset
sf.load_hf_dataset(
    train_path = '../data/scryfall_summarization_train.json',
    val_path = '../data/scryfall_summarization_val.json',
    test_path = '../data/scryfall_summarization_test.json'
)

Scryfall Tag Question Answering Dataset Loaded
	Train Records = 392
	Val Records = 98
	Test Records = 1


## Modeling

### With Trainer API  
source = https://huggingface.co/learn/llm-course/en/chapter7/5

In [ ]:
# fine tune the model

## instantiate the model
finetune = FineTuneLLM(
    model_name = MODEL
)

## prepare data
finetune.prepare_data(
    dataset = sf.dataset,
    max_input_length = MAX_INPUT_LENGTH,
    max_target_length = MAX_TARGET_LENGTH,
    batch_size = BATCH_SIZE
)

## train the model
finetune.train(
    learning_rate = LEARNING_RATE,
    accelerator_mixed_precision = 'fp16',
    accelerator_force_cpu = True,
    accelerator_gradient_steps = GRAD_ACCUMULATION_STEPS,
    num_train_epochs = NUM_EPOCHS
)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
c:\Users\nccru\anaconda3\envs\personal-general\Lib\site-packages\transformers\convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Map:   0%|          | 0/392 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

  0%|          | 0/3920 [00:00<?, ?it/s]

In [ ]:
# upload the model to the huggingface hub

## define the repo locally
## NOTE: Be sure to create OUTPUT_DIR in the hub manually first
repo_name = get_full_repo_name(OUTPUT_DIR)
repo = Repository(OUTPUT_DIR, clone_from = repo_name)

## save to hub
finetune.save_to_huggingface_hub(
    output_dir = OUTPUT_DIR,
    repo = repo,
    commit_message = f'Fine-tuned {MODEL} on scryfall tags.'
)

In [ ]:
assert 1 == 0

AssertionError: 

In [ ]:
# # load model
# # question_answerer = pipeline('question-answering', model = MODEL)
# # summarizer = pipeline('summarization', model = MODEL)
# summarizer = AutoModelForSeq2SeqLM.from_pretrained(MODEL)

# # get the tokenizer
# tokenizer = AutoTokenizer.from_pretrained(MODEL)
# print(f'Fast Tokenizer: {tokenizer.is_fast}')

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
c:\Users\nccru\anaconda3\envs\personal-general\Lib\site-packages\transformers\convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Fast Tokenizer: True


In [ ]:
# # truncated docs and summaries to fit the model
# from src.fine_tuning.preprocess import preprocess
# tokenized_datasets = sf.dataset.map(
#     lambda ex: preprocess(
#         tokenizer = tokenizer, 
#         examples = ex, 
#         max_input_length = MAX_INPUT_LENGTH, 
#         max_target_length = MAX_TARGET_LENGTH
#     ),
#     batched = True,
#     remove_columns = sf.dataset['train'].column_names
# )

Map:   0%|          | 0/392 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

In [ ]:
# # !pip install rouge_score
# import evaluate
# rouge_score = evaluate.load('rouge')

In [ ]:
# # collate data for mt5-small since it expects an encoder-decoder set up
# from transformers import DataCollatorForSeq2Seq
# data_collator = DataCollatorForSeq2Seq(tokenizer, model = summarizer, padding = True)

### With Accelerate

In [ ]:
# torch.cuda.is_available()

In [ ]:
# tokenized_datasets.set_format('torch')

In [ ]:
# # set up the data loader
# from torch.utils.data import DataLoader

# batch_size = 1 # 8 for full training
# train_dataloader = DataLoader(
#     tokenized_datasets['train'],
#     shuffle = True,
#     collate_fn = data_collator,
#     batch_size = batch_size
# )

# eval_dataloader = DataLoader(
#     tokenized_datasets['test'],
#     collate_fn = data_collator,
#     batch_size = batch_size
# )

In [ ]:
# # define our optimizer
# from torch.optim import AdamW
# optimizer = AdamW(summarizer.parameters(), lr = 2e-5)

In [ ]:
# # prepare the accelerator
# from accelerate import Accelerator

# accelerator = Accelerator(
#     mixed_precision = 'fp16', # if training on local machine
#     cpu = True, # if on local machine,
#     gradient_accumulation_steps = 8
# )
# summarizer, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
#     summarizer, optimizer, train_dataloader, eval_dataloader
# )

In [ ]:
# # define learning rate scheduler
# from transformers import get_scheduler

# num_train_epochs = 10
# num_update_steps_per_epoch = len(train_dataloader)
# num_training_steps = num_train_epochs * num_update_steps_per_epoch

# lr_scheduler = get_scheduler(
#     'linear',
#     optimizer = optimizer,
#     num_warmup_steps = 0,
#     num_training_steps = num_training_steps
# )

In [ ]:
# # define postprocesing
# import nltk
# def postprocess_text(preds, labels):
#     preds = [pred.strip() for pred in preds]
#     labels = [label.strip() for label in labels]

#     # ROUTE expects a new line after each sentence
#     preds = ['\n'.join(nltk.sent_tokenize(pred)) for pred in preds]
#     labels = ['\n'.join(nltk.sent_tokenize(label)) for label in labels]
    
#     return preds, labels

In [ ]:
# from huggingface_hub import Repository
# output_dir = 'scryfall-tag-summarizer'
# repo = Repository(output_dir, clone_from = repo_name)

In [ ]:
# from tqdm.auto import tqdm
# import torch
# import numpy as np

# progress_bar = tqdm(range(num_training_steps))

# for epoch in range(num_train_epochs):
#     # Training
#     summarizer.train()
#     for step, batch in enumerate(train_dataloader):
#         with accelerator.accumulate(summarizer):
#             outputs = summarizer(**batch)
#             loss = outputs.loss
#             accelerator.backward(loss)

#             optimizer.step()
#             lr_scheduler.step()
#             optimizer.zero_grad()
#             progress_bar.update(1)

#     # Evaluation
#     summarizer.eval()
#     for step, batch in enumerate(eval_dataloader):
#         with torch.no_grad():
#             generated_tokens = accelerator.unwrap_model(summarizer).generate(
#                 batch["input_ids"],
#                 attention_mask=batch["attention_mask"],
#             )

#             generated_tokens = accelerator.pad_across_processes(
#                 generated_tokens, dim=1, pad_index=tokenizer.pad_token_id
#             )
#             labels = batch["labels"]

#             # If we did not pad to max length, we need to pad the labels too
#             labels = accelerator.pad_across_processes(
#                 batch["labels"], dim=1, pad_index=tokenizer.pad_token_id
#             )

#             generated_tokens = accelerator.gather(generated_tokens).cpu().numpy()
#             labels = accelerator.gather(labels).cpu().numpy()

#             # Replace -100 in the labels as we can't decode them
#             labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
#             if isinstance(generated_tokens, tuple):
#                 generated_tokens = generated_tokens[0]
#             decoded_preds = tokenizer.batch_decode(
#                 generated_tokens, skip_special_tokens=True
#             )
#             decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

#             decoded_preds, decoded_labels = postprocess_text(
#                 decoded_preds, decoded_labels
#             )

#             rouge_score.add_batch(predictions=decoded_preds, references=decoded_labels)

#     # Compute metrics
#     result = rouge_score.compute()
#     # Extract the median ROUGE scores
#     result = {key: value * 100 for key, value in result.items()}
#     result = {k: round(v, 4) for k, v in result.items()}
#     print(f"Epoch {epoch}:", result)

#     # Save and upload
#     accelerator.wait_for_everyone()
#     unwrapped_model = accelerator.unwrap_model(summarizer)
#     unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)
#     if accelerator.is_main_process:
#         tokenizer.save_pretrained(output_dir)
#         repo.push_to_hub(
#             commit_message=f"Training in progress epoch {epoch}", blocking=False
#         )

## Use Fine-Tuned Model

In [ ]:
# load model from the hub
from transformers import pipeline
repo_name = get_full_repo_name(OUTPUT_DIR)
repo = Repository(OUTPUT_DIR, clone_from = repo_name)
summarizer = pipeline('summarization', model = repo)

In [ ]:
def print_summary(idx):
    card = sf.test['train'][idx]['document']
    actual_tags = sf.test['train'][idx]['summary']
    pred_tags = summarizer(sf.test['train'][idx]['document'])[0]['summary_text']

    print(f'Card = {card}\nActual Tags = {actual_tags}\nPredicted Tags = {pred_tags}')

for i in range(5):
    print(f"\n{'-' * 25}")
    print_summary(i)
    print(f"{'-' * 25}\n")

## Citations

@inproceedings{sanh2019distilbert,
  title={DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter},
  author={Sanh, Victor and Debut, Lysandre and Chaumond, Julien and Wolf, Thomas},
  booktitle={NeurIPS EMC^2 Workshop},
  year={2019}
}